# KonfAI MRSegmentator Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/MRSegmentator/MRSegmentator_demo.ipynb)

This notebook is meant to work from a **fresh environment**, including **Google Colab**.

[`mrsegmentator-konfai`](https://github.com/fideus-labs/KonfAI) is a thin KonfAI **app** wrapper for multi-organ **MR segmentation**:
it runs a published model from the Hugging Face repo [`VBoussot/MRSegmentator-KonfAI`](https://huggingface.co/VBoussot/MRSegmentator-KonfAI)
through the KonfAI runtime (patch-based, lazy inference — no full volume loaded into RAM).

It walks through:

- bootstrapping the app if the runtime is empty,
- downloading one public demo case (a MR volume),
- inspecting it,
- reviewing the exact `mrsegmentator-konfai segment` command,
- optionally running inference (toggle-gated — the model is downloaded from the Hub on first use).

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples").exists():
            return candidate
    return None


if IN_COLAB:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = find_repo_root(Path.cwd().resolve())
    if REPO_DIR is None:
        raise RuntimeError(
            "Could not locate the KonfAI repository. Run this notebook from the repo or open it in Google Colab."
        )

# Install the app CLI (and plotting deps) if missing.
for module, package in [("mrsegmentator_konfai", str(REPO_DIR / "apps" / "mrsegmentator")), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

EXAMPLE_DIR = REPO_DIR / "examples" / "MRSegmentator"
DATASET_DIR = EXAMPLE_DIR / "Dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_DIR)
print("Example dir:", EXAMPLE_DIR)

## 1. Download one public demo case

In [ ]:
import shutil

RUN_DOWNLOAD = True

if RUN_DOWNLOAD:
    subprocess.run([
        "hf", "download", "VBoussot/konfai-demo",
        "--repo-type", "dataset",
        "--include", "Synthesis/**",
        "--local-dir", str(DATASET_DIR),
    ], check=True)
    nested = DATASET_DIR / "Synthesis"
    if nested.exists():
        for item in nested.iterdir():
            target = DATASET_DIR / item.name
            if target.exists():
                shutil.rmtree(target) if target.is_dir() else target.unlink()
            shutil.move(str(item), str(target))
        shutil.rmtree(nested)
    shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

cases = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()) if DATASET_DIR.exists() else []
print("cases:", len(cases), "| first:", cases[0].name if cases else None)

## 2. Inspect the input MR volume

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

cases = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()) if DATASET_DIR.exists() else []
if not cases:
    raise RuntimeError("Dataset is empty. Run the download cell above first.")

case = cases[0]
INPUT_PATH = case / "MR.mha"
image = sitk.ReadImage(str(INPUT_PATH))
array = sitk.GetArrayFromImage(image).astype(np.float32)
print("case:", case.name, "| shape:", array.shape, "| spacing:", tuple(round(s, 2) for s in image.GetSpacing()))

mid = array.shape[0] // 2
lo, hi = np.percentile(array, [1, 99])
plt.figure(figsize=(4.5, 4.5))
plt.imshow(np.clip(array[mid], lo, hi), cmap="gray")
plt.title(f"MR — {case.name} (axial mid-slice)")
plt.axis("off")
plt.show()

## 3. Review the `mrsegmentator-konfai segment` command

In [ ]:
import torch

SELECTION = None  # single model; the number of folds to ensemble is the -f knob below
OUTPUT_DIR = EXAMPLE_DIR / "Output"
device_args = ["--gpu", "0"] if torch.cuda.is_available() else ["--cpu", "1"]

INFER_CMD = [
    "mrsegmentator-konfai", "segment",
        "-i", str(INPUT_PATH),
        "-o", str(OUTPUT_DIR),
        *device_args,
        "-f", "3",  # ensemble 3 cross-validation folds
]
print("$", " ".join(INFER_CMD))

## 4. Optionally run inference

In [ ]:
RUN_INFER = False  # set True to run — the model is downloaded from the Hub on first use, and a GPU is recommended

if RUN_INFER:
    try:
        subprocess.run(INFER_CMD, check=True)
        print("\nDone ->", OUTPUT_DIR)
    except (FileNotFoundError, subprocess.CalledProcessError) as error:
        print("Inference did not run:", error,
              "\nEnsure `mrsegmentator-konfai` is installed (pip install apps/mrsegmentator), a GPU is available,",
              "and the Hub `VBoussot/MRSegmentator-KonfAI` is reachable.")

## 5. Expected outputs

`mrsegmentator-konfai segment` writes a multi-organ label map under `Output/`, one file per input case (KonfAI
reassembles the patch-wise prediction with overlap blending).

Next steps:

- **evaluate** against a reference — `mrsegmentator-konfai eval -i mr.mha --gt reference.mha -o Output`;
- **pipeline** — `mrsegmentator-konfai pipeline ...` runs segment + eval (+ uncertainty) in one command;
- run on your own volumes — any `.mha` / `.nii.gz`, or an OME-Zarr / DICOM directory (KonfAI auto-detects the
  store format on read).